# Process Multiple Long Strips

In [ ]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import papermill

In [ ]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [ ]:
#%% Test Record Excel
fname_excel = r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx';
dftests = pd.read_excel(
    fname_excel,
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

In [ ]:
#%% Filter The Tests To Process
dfmasks = [
    #(dftests['Test date']=='2025-06-11') | (dftests['Test date']=='2025-06-12'),
    
    #dftests['Test date']=='2025-07-11',
    
    #dftests['Test ID'].isin([47,65]) # strips flagged as bad from May batch

    # strips flagged as "extra bad" from June 09 Batch
    #dftests['Test ID'].isin([73,77,85,88,97,117])
    #dftests['Test ID'].isin([97]),

    #dftests['Test name']!='GHL_pyapp_20250512T',
    #dftests['ProcessingNotes'].str.startswith('done,2'),
    
    dftests['Batch'].str.contains('July17n18Oldstriper') | dftests['Batch'].str.contains('July11WaxrobotStrips')
    
    #dftests['Batch'].str.contains('Valve')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

In [ ]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


In [ ]:
# Master Parameters
oct_scalar_min = 30;
oct_scalar_max = 60;

In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print(octstudy)

    fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    
    if(octstudy.study_info['study_num_oct_files']>0):
        # --Loading And Pre-Processing--
        if(fname_merged_and_rescaled_volume.exists()):
            pass;
            #print(f'Loading {fname_merged_and_rescaled_volume.name}')
            # Load the strip and merge into one volume
            #vdvol = vedo.Volume(pv.read(fname_merged_and_rescaled_volume));
            #octstudy.vdvol = vdvol;
        else:
            # Load OCT Data for this study
            octstudy.load_all_octs();


            # THESE WILL DO NOTHING IF ANTICIPATED OUTPUTS/ARTIFACTS ALREADY EXIST IN THE PROCESSED FOLDER

            # RGB Camera Images - Write them out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

            # RGB Camera Images - Make a montage and write out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_montage_image(octstudy);
            #break;


            # IF MERGED STACKED AND RESCALED TO SCALAR RANGE FILE EXISTS, LOAD THAT; OTHERWISE PROCESS IT HERE
            octstudy.folder_study_processed.mkdir(exist_ok=True);
            fname = fname_merged_and_rescaled_volume;
            if(fname.exists()):
                print('Stacked volume byte-size .vtk file already exists, will not recreate.');
                print(fname);
            else:
                print(f'Generating merged and rescaled .vtk volume');
                # Generate merged and rescaled volume
                vdvol = octstudy.generate_merged_vdvol_and_rescaled(oct_scalar_min,oct_scalar_max);
                octstudy.vdvol = vdvol;
            
                # save this byte-adjusted volume
                vdvol.dataset.save(fname);
                        
            # Unload OCT Data for this study (we will still keep the vdvol)
            octstudy.unload_all_octdata();

        # --Detailed Image Processing--
        if hasattr(octstudy,'vdvol'):
            del octstudy.vdvol;

# (util) delete some items from processed folder

In [ ]:
# Warning this can be destructive deleting processed data!!
if False:
    for idx,octstudy in enumerate(octstudies):
        print('~~~~~~~');
        print(octstudy.name);
        
        if False:
            fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
            if(fname.exists()):
                print('deleted',fname.name);
                fname.unlink();
            
            fname = (octstudy.folder_study_processed/'data_extracted.npz')
            if(fname.exists()):
                print('deleted',fname.name);
                fname.unlink();

            flist = octstudy.folder_study_processed.glob('figout*');
            for fname in flist:
                print('deleted',fname.name);
                fname.unlink();

            flist = octstudy.folder_study_processed.glob('processing_step*');
            for fname in flist:
                print('deleted',fname.name);
                fname.unlink();

        if False:
            # move files to a backup folder in the processed directory

            folder_backup = (octstudy.folder_study_processed/'_backup20250708');
            folder_backup.mkdir(exist_ok=True);

            fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
            if(fname.exists()):
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);
        
            fname = (octstudy.folder_study_processed/'data_extracted.npz')
            if(fname.exists()):
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);

            flist = octstudy.folder_study_processed.glob('figout*');
            for fname in flist:
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);

            flist = octstudy.folder_study_processed.glob('processing_step*');
            for fname in flist:
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);
        #break;


# (util) update excel index file

In [ ]:
# Allow update of excel index file?
import openpyxl
workbook = None;
if True:
    # directly open excel file
    workbook = openpyxl.load_workbook(fname_excel);

    # open the proper sheet in the excel file
    #sheet = workbook.active
    sheet = workbook['Strip measurement'];

    # get the header
    hdrrow = 2;
    hdrcols = sheet[hdrrow];
    print(hdrcols)
    hdr = {c.value:cnt+1 for cnt,c in enumerate(hdrcols)};
    print(hdr)

    # go through each row, and look for matching octstudy object we have loaded in memory
    madeChanges = False;
    for row in sheet.iter_rows(min_row=hdrrow+3,min_col=hdrcols[0].column):
        record = {hdrcols[cnt].value:c.value for cnt,c in enumerate(row)}
        matchingoctstudy = [octstudy for octstudy in octstudies if octstudy.name==record["Test name"]]
        if len(matchingoctstudy)==1:
            print(record)
            print(octstudy.name)
            octstudy = matchingoctstudy[0]
            octstudy.load_first_oct();
            octdata = octstudy.octdatalist[0]
            cfg_oct_xml = octdata.cfg_oct_xml;

            # populate filesize
            row[hdr['Total size (GB)']-1].value = round(sum([f.stat().st_size for f in octstudy.folder_study.glob('*.oct')])/1e9,2);
            row[hdr['File size (MB)']-1].value = round((sum([f.stat().st_size for f in octstudy.folder_study.glob('*.oct')])/1e9)/octstudy.num_oct_files,2);

            # populate acquisition details
            pixelsize_x = float(cfg_oct_xml.Ocity.Image.PixelSpacing.SizeX.etElem.text)
            pixelsize_y = float(cfg_oct_xml.Ocity.Image.PixelSpacing.SizeY.etElem.text)
            pixelsize_z = float(cfg_oct_xml.Ocity.Image.PixelSpacing.SizeZ.etElem.text)
            totalsizemm_x = float(cfg_oct_xml.Ocity.Image.SizeReal.SizeX.etElem.text)
            totalsizemm_y = float(cfg_oct_xml.Ocity.Image.SizeReal.SizeY.etElem.text)
            totalsizemm_z = float(cfg_oct_xml.Ocity.Image.SizeReal.SizeZ.etElem.text)
            row[hdr['X, FOV']-1].value = totalsizemm_x;
            row[hdr['Y, FOV']-1].value = totalsizemm_y;
            row[hdr['Z, FOV']-1].value = totalsizemm_z;
            row[hdr['X, pixel size']-1].value = pixelsize_x*1e3;
            row[hdr['Y, pixel size']-1].value = pixelsize_y*1e3;
            row[hdr['Z, pixel size']-1].value = pixelsize_z*1e3;
            row[hdr['Angle']-1].value = float(cfg_oct_xml.Ocity.Image.Angle.etElem.text)
            row[hdr['Speed/Sensitivity']-1].value = cfg_oct_xml.Ocity.Instrument.DevicePresetDescription.etElem.text;
            row[hdr['Averaging (A-scan)']-1].value = int(cfg_oct_xml.Ocity.Acquisition.IntensityAveraging.AScans.etElem.text)
            row[hdr['Refractive Index']-1].value = round(float(cfg_oct_xml.Ocity.Acquisition.RefractiveIndex.etElem.text),1);

            if octstudy.has_json_info_file:
                jsoninfo = octstudy.json_info_file;
                row[hdr['Left edge (mm)']-1].value = jsoninfo['pos_leftmost_edge_center'];
                row[hdr['Right edge (mm)']-1].value = jsoninfo['pos_rightmost_edge_center'];
                row[hdr['Strip length (mm)']-1].value = jsoninfo['pos_rightmost_edge_center']-jsoninfo['pos_leftmost_edge_center'];

            # rescheck = octstudy.resultsCheck();
            # haveAllResults = not all([v is False for k,v in rescheck.items()])
            # #haveAllResults = haveAllResults and (len(rescheck['along_strip_data_extracted'])==5)
            # print('results done?',haveAllResults)
            # if(haveAllResults):
            #     row[hdr['ProcessingNotes']-1].value = 'done,{:d}'.format(len(rescheck['along_strip_data_extracted']))
            # else:
            #     row[hdr['ProcessingNotes']-1].value = ''
            # #break

            madeChanges = True

    if(madeChanges):
        # make a copy of the excel index
        fname_excel_copy = Path(fname_excel).parent/(Path(fname_excel).stem + time.strftime('_bak_%Y%m%dT%H%M.xlsx'))
        import shutil
        shutil.copy2(Path(fname_excel),fname_excel_copy);

        # then save the modified workbook on top of the old excel file
        workbook.save(fname_excel)
    else:
        print('No changes to excel index file made.');

    workbook.close();
    

# Call Notebooks - Processing A to D and E

In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
norunlist = [
    #'GHL_pyapp_20250508T1112','GHL_pyapp_20250508T1128','GHL_pyapp_20250508T1134','GHL_pyapp_20250508T1143'
]
forcerunlist = [
    #'GHL_pyapp_20250508T1151',
    #'GHL_pyapp_20250508T1601',
    #'GHL_pyapp_20250508T1608',
]
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    rescheck = octstudy.resultsCheck();
    pp.pprint(rescheck)
    doWeRunTheNotebook = any([v is False for k,v in rescheck.items()])
    #doWeRunTheNotebook = doWeRunTheNotebook or ('/dfstepE' not in rescheck['along_strip_data_extracted']);
    doWeRunTheNotebook = doWeRunTheNotebook or ('/dfstepA' not in rescheck['along_strip_data_extracted']);
    print('Run?',doWeRunTheNotebook)
    if((doWeRunTheNotebook and octstudy.name not in norunlist) or (octstudy.name in forcerunlist)):
        octstudies_to_run.append(octstudy);
    # if(octstudy.name in forcerunlist):
    #     octstudies_to_run.append(octstudy);

# if(len(octstudies_to_run)>4):
#     #octstudies_to_run=octstudies_to_run[0:4];
#     #octstudies_to_run=octstudies_to_run[-4:];
#     print('only a subset');
#     norunlist = [
#         'GHL_pyapp_20250508T1112','GHL_pyapp_20250508T1128','GHL_pyapp_20250508T1134','GHL_pyapp_20250508T1143'
#     ]
#     octstudies_to_run = [s for s in octstudies_to_run if s.name not in norunlist ]
#     pass;

#octstudies_to_run = octstudies_to_run[0:20];

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

# Prepare notebooks we will call

In [ ]:
nbpaths = [
    #r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250615_process_a_longstrip.ipynb",
    #r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250721_process_a_longstrip.ipynb",
    r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250722_process_a_longstrip.ipynb",
];
for nbpath in nbpaths:
    print('Notebook:',Path(nbpath).name);
    parameters = papermill.inspect_notebook(nbpath)
    pp.pprint(parameters.keys());

# Call notebooks to process (Multiple-strips in parallel, concurrent futures)

In [ ]:
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):

    folder_temp_path = Path(r'D:\TEMP\OCTtmp');
    folder_temp_path.mkdir(parents=True,exist_ok=True);

    # run each notebook we have defined to be run
    nnotebooks = len(nbpaths);
    for count,nbpath in enumerate(nbpaths):
        nbpath = Path(nbpath)
        nbpath_out = folder_temp_path/(nbpath.stem+'_OUT_{:s}.ipynb').format(octstudy.name)

        # execute a notebook
        print(f'Launching for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} ...')
        try:
            papermill.execute_notebook(
                input_path=nbpath,
                output_path=nbpath_out,
                parameters=dict(
                    folder_octexport_root=folder_octexport_root.as_posix(),
                    codename = Path(nbpath).name,
                    study_name=octstudy.name,
                    folder_figure_temp=nbpath_out.parent.as_posix()
                )
            )
            print(f'Finished for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} !')
        except Exception as e:
            print(f'FAILED for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} !')
            break;
    print(f'alldone for {octstudy.name}')


# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result());

# Example usage
num_processes = 6;  # Number of parallel processes
run_in_parallel(num_processes);